# Datamine TRIVAL Process Example

This notebook demonstrates how to configure and run the **`trival`** process wrapper in `dmstudio`.

## Process Description

## TRIVAL

Evaluates a standard Datamine block model against a triangulated wireframe model, reporting on the contents of closed wireframes or the reserves below or above a single-surface (digital terrain model) wireframe.

Trival generates a **RESULTS** file of reserves from an orebody model and an input wireframe; and optionally produces an output orebody model in which a proportion **MINED** field is updated. Evaluation is normally by partial block, but can optionally be done on a full-block basis in which any block with a **MINED** proportion greater than 0.5 is marked as wholly **MINED** , any proportion less than 0.5 is treated as zero. Results can be classified by XY, XZ, or YZ planes. The results file can be read by [TABRES](<tabres.md>) to produce various reserve tabulations.

The input wireframe triangle file must contain an **SID** (Surface IDentification) field. If an 'upper' digital terrain model is supplied (with **SID** values all +1) then the model created will fill cells below the surface. Inversely, if all **SID** values are -1, then cell filling will be done above the surface. For solid wireframe models (in which **SID** values of +1 represent upward facing surfaces and -1 represent lower facing surfaces), cells are filled inside the wireframe. Any number of separate zones may be handled within one run of **TRIVAL** , provided they are all held within the same pair of &**WIREPT** , &**WIRETR** files. The SID field is produced automatically by **TRIANR** (set to +1 or -1 at user option).

### Input and Output Models

The input model is in standard Datamine model file format, and may contain cells and sub-cells, in any order; the model does not have to be sorted on IJK to be processed by **TRIVAL**. The fraction of the cell or sub-cell mined may be stored in the **MINED** field of the optional output model file, as a value in the range 0 to 1. The output model file will be an exact copy of the input file, with the **MINED** field added if it did not already exist in the input file. The output model file may be the same as the input model file (in-place updating) if the **MINED** field existed in the input file.

### Balancing Volumes

**TRIVAL** produces for each pass through the model a balancing volume record in the results file; the balancing volume recorded (NOT the same as that produced by **[MODRES](<modres.md>)**) is the total volume of the model less the volume intersected by the wireframe zone evaluated.

### Densities

In computation of tonnages from volumes, any **DENSITY** field within the input model is used. If the **DENSITY** field exists, then unmodelled regions will use the default value of the **DENSITY** field. If the **DENSITY** field does not exist, then an overall density may be supplied by use of the @**DENSITY** parameter. If this was not supplied, then the density used is 1.0.

### Passes through the Model File

The process reads in one wireframe zone at a time (as identified by the **ZONE** field if present in the triangle file). For each wireframe zone **TRIVAL** reads through the entire model and records its evaluation against each separately. If successive zones overlap, or the input model already contains **MINED** values greater than 0, there are two ways in which the evaluation may be recorded in the **RESULTS** file and the output **MINED** field.

**Note** : Up to 2000 unique zone values can exist in the specified **ZONE** field.

These two options are controlled by the supplied value of the optional @**INCRMENT** parameter. Either it will be considered that all mining is additive if @**INCRMENT** is set to 0, (i.e. that there are no overlapping zones), in which case any cell already partly mined will have the full volume of the current evaluation added to the amount mined - subject to a maximum proportion mined of 1.0. Alternatively, if @**INCRMENT** is set to 1, it will be considered that mining is incremental, and that successive zones are to be considered as mine expansions, and overlap totally. In this case, a partly mined cell will be output with a proportion mined which is the maximum of the previous and current proportions. In this case, naturally, if the current proportion mined is less than the previous, there will be no evaluated volume in the results file for the cell.

### Input Files:

* **modeli** (Block Model):
  Model file for evaluation. Must contain at least the fields XC, YC, ZC, XINC, YINC,

## ZINC, XMORIG, YMORIG, ZMORIG, NX, NY, NZ, IJK.

  Required=Yes

* **wiretr** (Wireframe Triangle):
  Input wireframe triangle file.
  Required=Yes

* **wirept** (Wireframe Points):
  Input wireframe point file.
  Required=Yes

### Output Files:

* **results** (Undefined):
  The output results file, in a format suitable for input into the TABRES process
  Required=Yes

* **modelo** (Block Model):
  Optional output model file. This may be the same as the input, if the MINED field
  exists in the input file. The MINED field will be created in the output file if it does
  not exist.
  Required=No

### Fields:

* **zone** (a/n : IN):
  Optional zone identifier field (numeric or single-word alpha) in the WIRETR triangle
  file, defining individual wire-frame zone models. This field will be added to the
  results file. Up to 2000 zone values can exist in the specified column.
  Default=Undefined
  Required=No

### Parameters:

* **density**:
  Set required density value. This will only be used if there is no DENSITY field in the
  input model. If there is no DENSITY field, and no DENSITY parameter, then a value of 1.0
  is used.
  Range=
  Values=
  Default=
  Required=Yes

* **fullcell**:
  = 0 : partial cell evaluation = 1 : whole cell evaluation in place of partial cell
  evaluation (0)..
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **mine**:
  Optional; if non-zero, output proportion mined in MINED field of MODELO file (0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **plane**:
  Optional alpha parameter defining slice orientation to be used in classification of
  results 'XY', 'XZ', or 'YZ'.
  Range=
  Values=XY, YZ, XZ
  Default="XY"
  Required=No

* **incrment**:
  (0) If non-zero, mining is assumed to be incremental, and for any (sub-)cell the amount
  mined in any pass must be greater than the total mined so far for it to be recorded. If
  zero, mining is assumed to be additive, subject to a total mined fraction of 1.
  Range=0....1
  Values=0,1
  Default=0
  Required=No

* **modltype**:
  Type of wireframe model to be evaluated; one of the following options:- =1 : solid 3d,
  interior to be evaluated. =3 : surface, cells to be evaluated below [for XY], to south
  [for XZ], or to west [for YZ]. =4 : surface, cells to be evaluated above [for XY], to
  north [for XZ], or to east [for YZ].
  Range=
  Values=1, 3, 4
  Default=0
  Required=No

* **chkovlap**:
  Control checking of overlapping triangles. This applies only to model types of 3 and 4.
  If set to 1 then checking of triangle overlaps is performed. If set to zero then no
  checking will occur. Checking should be done where DTMs curve over themselves as in some
  seam models. Checking can be turned off for DTMs where this is known not to occur, such
  as in an open pit wireframe. If checking is turned off the process will run faster. (1).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **checkrot**:
  Set to 1 to automatically check for and correctly process rotated models. The default
  value is zero. If this is not set to 1 and the model is rotated then the wireframe
  points need to be transformed into the model coordinate space using CDTRAN before
  running TRIFIL.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **nosid**:

* **Options** (1: Disables checking of SIDs for increased speed. Use for single surfaces):
  only.
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **print**:
  =1 : ; Show a line for each cell evaluated in each perimeter.(0).
  Range=0,1
  Values=0,1
  Default=0
  Required=No

* **enter_evaluation_parameters**:
  Standard evaluation consists of mean grades and tonnages for each <bench or section>
  Enter Y or Yes (or <RETURN>) for standard evaluation: Enter N or No to specify
  evaluation parameters. >YES/NO > Y or Yes or <return> to use these; N or No to enter
  parameters.
  Range=
  Values=
  Default=
  Required=No

* **rocktype_classification**:
  Each evaluation (within a perimeter on a <bench or section> may be classified by
  rocktype. There are two types of classification possible:- By ore, waste, air, stockpile
  etc. or by given values of the rocktype field: Results may only be classified by
  rocktype if there is a suitable rocktype field. Enter Y or Yes (or <RETURN>) for
  classification. Enter N or No to omit classification by rocktype
  Range=
  Values=
  Default=
  Required=No

* **grade_intervals**:
  If no grade intervals are specified then means are calculated. Otherwise you may
  specify grade ranges of a named numeric field. Enter Y or Yes (or <RETURN>) to specify
  grade ranges. Enter N or No to compute means only. If Y, Yes or <return>; Enter name of
  main grade field. >FIELD > Enter field name.
  Range=
  Values=
  Default=
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('trival')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute trival
print("Running trival...")
dm_cmd.trival(
    modeli_i='t_mod1',  # required
    wiretr_i='t_SurfaceTriangles',  # required
    wirept_i='t_SurfaceTriangles',  # required
    results_o=['t_trival_out'],  # required
    density_p='required_val',  # required
    # modelo_o='t_trival_out',  # optional
    # zone_f='optional',  # optional
    # fullcell_p=0,  # optional
    # mine_p=0,  # optional
    # plane_p='XY',  # optional
    # incrment_p=0,  # optional
    # modltype_p=0,  # optional
    # chkovlap_p=0,  # optional
    # checkrot_p=0,  # optional
    # nosid_p=0,  # optional
    # print_p=0,  # optional
    # enter_evaluation_parameters_p='optional',  # optional
    # rocktype_classification_p='optional',  # optional
    # grade_intervals_p='optional',  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("trival execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_trival_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")